In [1]:
import pandas as pd

In [2]:
#demand data
year_url_map={
    2025:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/b2bde559-3455-4021-b179-dfe60c0337b0/download/demanddata_2025.csv",
    2024:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/f6d02c0f-957b-48cb-82ee-09003f2ba759/download/demanddata_2024.csv",
    2023:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/bf5ab335-9b40-4ea4-b93a-ab4af7bce003/download/demanddata_2023.csv",
    2022:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/bb44a1b5-75b1-4db2-8491-257f23385006/download/demanddata_2022.csv",
    2021:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/18c69c42-f20d-46f0-84e9-e279045befc6/download/demanddata_2021.csv",
    2020:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/33ba6857-2a55-479f-9308-e5c4c53d4381/download/demanddata_2020.csv",
    2019:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/dd9de980-d724-415a-b344-d8ae11321432/download/demanddata_2019.csv",
    2018:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/fcb12133-0db0-4f27-a4a5-1669fd9f6d33/download/demanddata_2018.csv",
    2017:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/2f0f75b8-39c5-46ff-a914-ae38088ed022/download/demanddata_2017.csv",
    2016:"https://api.neso.energy/dataset/8f2fe0af-871c-488d-8bad-960426f24601/resource/3bb75a28-ab44-4a0b-9b1c-9be9715d3c44/download/demanddata_2016.csv"
}

In [3]:
#unify settlement_date into standardization form
dfs=[]

for year,url in year_url_map.items():
    try:
        df=pd.read_csv(url)
        dfs.append(df)
        print(f'{year} is done')
    except Exception as e:
        print(f'{year}:{url} not downloaded',e)

2025 is done
2024 is done
2023 is done
2022 is done
2021 is done
2020 is done
2019 is done
2018 is done
2017 is done
2016 is done


In [4]:
df=pd.concat(dfs)
df=df.drop(columns=['SETTLEMENT_DATE'])
df=df.drop(columns=['SETTLEMENT_PERIOD'])
generate_time=pd.date_range(start='2016-01-01 00:00:00',end='2025-12-31 23:30:00',freq='30min')

if len(generate_time)==len(df.index):
    df['time']=generate_time
else:
    print(len(df.index))
    print(len(generate_time))

print(df.tail())
print(df.shape)
print(df.columns)

          ND    TSD  ENGLAND_WALES_DEMAND  EMBEDDED_WIND_GENERATION  \
17563  29837  31233                 27097                      1735   
17564  28701  30304                 26042                      1742   
17565  28188  29800                 25464                      1750   
17566  27429  28126                 24730                      1694   
17567  27121  27739                 24502                      1637   

       EMBEDDED_WIND_CAPACITY  EMBEDDED_SOLAR_GENERATION  \
17563                    4871                          0   
17564                    4871                          0   
17565                    4871                          0   
17566                    4871                          0   
17567                    4871                          0   

       EMBEDDED_SOLAR_CAPACITY  NON_BM_STOR  PUMP_STORAGE_PUMPING  \
17563                    11503            0                     5   
17564                    11503            0                     5   
17565

In [5]:
from src.extract.save_dir import saved_data_dir
from src.transform.half_hour_to_hour import hh_to_hour

In [6]:
#change half_hourly data into hourly data
list_to_mean=[col for col in df.columns if col != 'time']
hh_to_hour('time',df,list_to_mean,'../data/raw/electricity_demand.csv')

variable time change to time, saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\raw\electricity_demand.csv


In [7]:
#weather_data
import requests

In [8]:
#api
url="https://archive-api.open-meteo.com/v1/archive"
params={
    "latitude": 51.5074,
    "longitude": -0.1278,
    'hourly': "temperature_2m,relative_humidity_2m,wind_speed_10m",
    "start_date": "2016-01-01",
    "end_date": "2025-12-31",
    "timezone": "Europe/London"
}

In [9]:
response=requests.get(url,params=params)
if response.status_code==200:
    data=response.json()
    print(data.keys())
    print(data['hourly'].keys())
    df=pd.DataFrame(data['hourly'])
    df['time']=pd.to_datetime(df['time'])
    saved_data_dir('london_10years.csv',df,output_dir='../data/raw')
else:
    print(response.status_code)

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m'])
london_10years.csv saved to C:\Users\24363\Desktop\powering_market_forecasting_analytics\data\raw\london_10years.csv
